In [3]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.sql import SparkSession
import seaborn as sns
import pandas as pd

# Start Spark Session
spark = SparkSession.builder.appName("IrisClassification").getOrCreate()

# Load Online Dataset (Iris dataset from seaborn)
iris = sns.load_dataset("iris")
iris_df = spark.createDataFrame(iris)

# Convert 'species' column to numeric labels
indexer = StringIndexer(inputCol="species", outputCol="species_index")
iris_df = indexer.fit(iris_df).transform(iris_df)

# Assemble features
assembler = VectorAssembler(inputCols=["sepal_length", "sepal_width", "petal_length", "petal_width"],
                            outputCol="features")
train_data = assembler.transform(iris_df).select("features", "species_index")

# Train Decision Tree Classifier
dt = DecisionTreeClassifier(labelCol="species_index", featuresCol="features")
dt_model = dt.fit(train_data)

# Show results
dt_model.transform(train_data).show(5)


+-----------------+-------------+--------------+-------------+----------+
|         features|species_index| rawPrediction|  probability|prediction|
+-----------------+-------------+--------------+-------------+----------+
|[5.1,3.5,1.4,0.2]|          0.0|[50.0,0.0,0.0]|[1.0,0.0,0.0]|       0.0|
|[4.9,3.0,1.4,0.2]|          0.0|[50.0,0.0,0.0]|[1.0,0.0,0.0]|       0.0|
|[4.7,3.2,1.3,0.2]|          0.0|[50.0,0.0,0.0]|[1.0,0.0,0.0]|       0.0|
|[4.6,3.1,1.5,0.2]|          0.0|[50.0,0.0,0.0]|[1.0,0.0,0.0]|       0.0|
|[5.0,3.6,1.4,0.2]|          0.0|[50.0,0.0,0.0]|[1.0,0.0,0.0]|       0.0|
+-----------------+-------------+--------------+-------------+----------+
only showing top 5 rows



In [13]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier, OneVsRest
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, RegressionEvaluator
from pyspark.sql import SparkSession
import seaborn as sns

# Initialize Spark Session
spark = SparkSession.builder.appName("IrisClassification").getOrCreate()
print("✅ Spark Session Created")

# Load dataset
iris = sns.load_dataset("iris")
iris_df = spark.createDataFrame(iris)
print("✅ Dataset Loaded")
iris_df.show(5)  # Print first 5 rows

# Convert species column to numerical labels
indexer = StringIndexer(inputCol="species", outputCol="species_index")
iris_df = indexer.fit(iris_df).transform(iris_df)
print("✅ Species Column Indexed")
iris_df.show(5)  # Check transformed dataset

# Assemble features
assembler = VectorAssembler(inputCols=["sepal_length", "sepal_width", "petal_length", "petal_width"],
                            outputCol="features")
train_data = assembler.transform(iris_df).select("features", "species_index")
print("✅ Features Assembled")
train_data.show(5, truncate=False)  # Print first 5 rows with full content

### Logistic Regression Model ###
lr = LogisticRegression(labelCol="species_index", featuresCol="features")

# Fit logistic regression
lr_model = lr.fit(train_data)
print("✅ Logistic Regression Model Trained")

# Corrected: Use coefficientMatrix for multinomial logistic regression
print("Coefficient Matrix:\n", lr_model.coefficientMatrix)
print("Intercept Vector:", lr_model.interceptVector)

### Random Forest Model ###
rf = RandomForestClassifier(labelCol="species_index", featuresCol="features")
rf_model = rf.fit(train_data)
print("✅ Random Forest Model Trained")

### Gradient Boosted Trees Model ###
gbt = GBTClassifier(labelCol="species_index", featuresCol="features")
ovr = OneVsRest(classifier=gbt, labelCol="species_index", featuresCol="features")
gbt_model = ovr.fit(train_data)
print("✅ Gradient Boosted Trees Model Trained")

### Linear Regression Model ###
lr_reg = LinearRegression(featuresCol="features", labelCol="species_index")
lr_reg_model = lr_reg.fit(train_data)
print("✅ Linear Regression Model Trained")
print("Coefficients:", lr_reg_model.coefficients)
print("Intercept:", lr_reg_model.intercept)







### Cross-Validation ###
evaluator = MulticlassClassificationEvaluator(labelCol="species_index", metricName="accuracy")
param_grid = ParamGridBuilder().addGrid(lr.regParam, [0.1, 0.01]).build()
crossval = CrossValidator(estimator=lr, estimatorParamMaps=param_grid, evaluator=evaluator, numFolds=3)
cv_model = crossval.fit(train_data)
print("✅ Cross-Validation Completed")

print("🎉 All models trained successfully!")


✅ Spark Session Created
✅ Dataset Loaded
+------------+-----------+------------+-----------+-------+
|sepal_length|sepal_width|petal_length|petal_width|species|
+------------+-----------+------------+-----------+-------+
|         5.1|        3.5|         1.4|        0.2| setosa|
|         4.9|        3.0|         1.4|        0.2| setosa|
|         4.7|        3.2|         1.3|        0.2| setosa|
|         4.6|        3.1|         1.5|        0.2| setosa|
|         5.0|        3.6|         1.4|        0.2| setosa|
+------------+-----------+------------+-----------+-------+
only showing top 5 rows

✅ Species Column Indexed
+------------+-----------+------------+-----------+-------+-------------+
|sepal_length|sepal_width|petal_length|petal_width|species|species_index|
+------------+-----------+------------+-----------+-------+-------------+
|         5.1|        3.5|         1.4|        0.2| setosa|          0.0|
|         4.9|        3.0|         1.4|        0.2| setosa|          0.0|

In [15]:
!wget https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv


--2025-03-16 17:23:51--  https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3858 (3.8K) [text/plain]
Saving to: ‘iris.csv’

iris.csv            100%[===================>]   3.77K  --.-KB/s    in 0s      

2025-03-16 17:23:51 (48.4 MB/s) - ‘iris.csv’ saved [3858/3858]



In [16]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

# ✅ Step 1: Create a Spark Session
spark = SparkSession.builder.appName("IrisClassification").getOrCreate()
print("✅ Spark Session Created")

# ✅ Step 2: Load the Dataset
iris_df = spark.read.csv("iris.csv", header=True, inferSchema=True)
print("✅ Dataset Loaded")
iris_df.show(5)  # Show first 5 rows

# ✅ Step 3: Check if species_index already exists
if "species_index" in iris_df.columns:
    iris_df = iris_df.drop("species_index")  # Drop it to avoid duplication

# ✅ Step 4: Convert Species Column to Numerical Index
indexer = StringIndexer(inputCol="species", outputCol="species_index")
print("✅ Species Column Indexed")

# ✅ Step 5: Assemble Features
assembler = VectorAssembler(inputCols=["sepal_length", "sepal_width", "petal_length", "petal_width"],
                            outputCol="features")
print("✅ Features Assembled")

# ✅ Step 6: Define and Train Logistic Regression Model
lr = LogisticRegression(featuresCol="features", labelCol="species_index")

# ✅ Step 7: Define the Pipeline
pipeline = Pipeline(stages=[indexer, assembler, lr])

# ✅ Step 8: Train the Pipeline Model
pipeline_model = pipeline.fit(iris_df)  # ✅ No more errors!
print("✅ Pipeline Created and Trained")


✅ Spark Session Created
✅ Dataset Loaded
+------------+-----------+------------+-----------+-------+
|sepal_length|sepal_width|petal_length|petal_width|species|
+------------+-----------+------------+-----------+-------+
|         5.1|        3.5|         1.4|        0.2| setosa|
|         4.9|        3.0|         1.4|        0.2| setosa|
|         4.7|        3.2|         1.3|        0.2| setosa|
|         4.6|        3.1|         1.5|        0.2| setosa|
|         5.0|        3.6|         1.4|        0.2| setosa|
+------------+-----------+------------+-----------+-------+
only showing top 5 rows

✅ Species Column Indexed
✅ Features Assembled
✅ Pipeline Created and Trained
